In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torch.nn as nn
from torch import Tensor
import yaml

from parnassus_core.flow.models.wrapper import FlowLightning
from parnassus_core.experiments.cms_2011.dataset import CMSDataset

In [ ]:
ckpt_path = "../logs/cms2011_event_fm_MSE_20250825-T154555/ckpts/last.ckpt"
cfg_path = "../logs/cms2011_event_fm_MSE_20250825-T154555/config.yaml"

In [ ]:
ckpt_path = "../logs/cms2011_particle_rcfm_MSEDir_Deep5M_Flex_Simple_20250825-T161212/ckpts/last.ckpt"
cfg_path = "../logs/cms2011_particle_rcfm_MSEDir_Deep5M_Flex_Simple_20250825-T161212/config.yaml"

In [ ]:
with open(cfg_path, "r") as f:
    cfg = yaml.safe_load(f)

In [ ]:
ds = CMSDataset(
    cfg["data"]["train_path"],
    max_particles=400,
    train_type="event",
    # train_type="particle",
    sin_cos=True,
    zero_neutral_vtx=True,
    scale_dict=cfg["data"]["scale_dict"],
    var_dict=cfg["data"]["var_dict"],
    reduce_ds=1000,
    mode="train",
)

In [ ]:
dl = torch.utils.data.DataLoader(
    ds,
    batch_size=20,
    shuffle=False,
)

In [ ]:
fs, ctxt = next(iter(dl))

In [ ]:
from parnassus_core.flow.export.utils import load_model_and_config, export, validate

In [ ]:
model, config = load_model_and_config(cfg_path, ckpt_path)

In [ ]:
fs_data, ctxt_dict = next(iter(dl))


def prepare_data(fs_data, ctxt_dict, mode="particle"):
    timestep = torch.rand(fs_data.shape[0], 1).to(fs_data.dtype)
    if mode == "particle":
        mask = torch.stack([ctxt_dict["ctxt_mask"], ctxt_dict["fs_mask"]], dim=-1)
    else:
        mask = ctxt_dict["ctxt_mask"]
    ctxt_data = ctxt_dict["ctxt_data"]
    ctxt_global_data = ctxt_dict["ctxt_global_data"]

    return {
        "fs_data": fs_data,
        "timestep": timestep,
        "ctxt_data": ctxt_data,
        "ctxt_global_data": ctxt_global_data,
        "mask": mask,
    }

In [ ]:
sample_batch = prepare_data(*(next(iter(dl))), mode=ds.train_type)

In [ ]:
model_wrapper, outpath = export(
    model,
    config,
    "onnx",
    "particle.onnx",
    sample_batch,
)

In [ ]:
import onnx

In [ ]:
onnx_model = onnx.load("particle.onnx")
print(onnx.helper.printable_graph(onnx_model.graph))

In [ ]:
validate(
    model_wrapper,
    outpath,
    sample_batch,
    "onnx",
)

In [ ]:
model_wrapper, outpath = export(
    model,
    config,
    "torch",
    "particle.pt2",
    sample_batch,
)

In [ ]:
dl = torch.utils.data.DataLoader(
    ds,
    batch_size=2,
    shuffle=False,
)

In [ ]:
sample_batch = prepare_data(*next(iter(dl)))

In [ ]:
print(torch.export.load("particle.pt2"))

In [ ]:
validate(
    model_wrapper,
    outpath,
    sample_batch,
    "torch",
    # device=torch.device("cuda")
)

In [ ]:
model = FlowLightning.load_from_checkpoint(
    ckpt_path,
    map_location=torch.device("cpu"),
)

In [ ]:
model.net.change_backend("torch")

In [ ]:
model.eval();

In [ ]:
from torch.export import Dim

In [ ]:
class ONNXModel(nn.Module):
    def __init__(
        self, net: nn.Module, dynamic_axes: dict[str, type[Dim] | dict[int, str]]
    ) -> None:
        super().__init__()
        self.net = net
        self.dynamic_axes = dynamic_axes

    @torch.no_grad()
    def forward(
        self,
        fs_data: Tensor,
        timestep: Tensor,
        mask: Tensor,
        ctxt_data: Tensor,
        ctxt_global_data: Tensor,
    ) -> Tensor:  # type: ignore[override]
        """Forward pass through the model.

        The arguments must be passed in the same order they appear in the feature map.
        """

        ctxt_dict: dict[str, torch.Tensor] = {
            "ctxt_data": ctxt_data,
            "ctxt_global_data": ctxt_global_data,
        }
        # if mask.ndim == 2:
        #     # If mask is a single value, we assume it's a ctxt mask
        # ctxt_dict["ctxt_mask"] = mask
        # elif mask.ndim > 2:
        #     # If mask is a sequence, we assume it's a fs mask
        ctxt_dict["fs_mask"] = mask[..., 0]
        ctxt_dict["ctxt_mask"] = mask[..., 1]
        outputs = self.net(fs_data, timestep, ctxt_dict)
        return outputs

In [ ]:
evt_onnx_dynamic_axes = {
    "fs_data": {0: "batch"},
    "timestep": {0: "batch"},
    "mask": {0: "batch"},
    "ctxt_data": {0: "batch"},
    "ctxt_global_data": {0: "batch"},
}

In [ ]:
particle_onnx_dynamic_axes = {
    "fs_data": {0: "batch"},
    "timestep": {0: "batch"},
    "mask": {0: "batch"},
    "ctxt_data": {0: "batch"},
    "ctxt_global_data": {0: "batch"},
}

In [ ]:
fs_data, ctxt_dict = next(iter(dl))

In [ ]:
timestep = torch.tensor([0.5] * fs_data.shape[0], dtype=torch.float32)
mask = torch.stack([ctxt_dict["ctxt_mask"], ctxt_dict["fs_mask"]], dim=-1)
ctxt_data = ctxt_dict["ctxt_data"]
ctxt_global_data = ctxt_dict["ctxt_global_data"]

In [ ]:
net_onnx = ONNXModel(
    model.net,
    dynamic_axes=particle_onnx_dynamic_axes,
)

In [ ]:
onnx_program = torch.onnx.export(
    net_onnx,
    (fs_data, timestep, mask, ctxt_data, ctxt_global_data),
    input_names=["fs_data", "timestep", "mask", "ctxt_data", "ctxt_global_data"],
    # output_names=['output'],
    dynamic_axes=net_onnx.dynamic_axes,
    f="test_particle_jit.onnx",
    opset_version=18,
    verbose=False,
    report=True,
    # verify=True,
    export_params=True,
)

In [ ]:
import onnx

In [ ]:
test_model = onnx.load("test_particle_jit.onnx")
onnx.checker.check_model(test_model, full_check=True)

In [ ]:
onnx.checker.check_graph(test_model.graph)

In [ ]:
batch = Dim("batch", min=1, max=4096)

In [ ]:
evt_torch_dynamic_axes = {
    "fs_data": {0: batch, 1: Dim.STATIC, 2: Dim.STATIC},
    "timestep": {0: batch},
    "mask": {0: batch, 1: Dim.STATIC},
    "ctxt_data": {0: batch, 1: Dim.STATIC, 2: Dim.STATIC},
    "ctxt_global_data": {0: batch, 1: Dim.STATIC},
}

In [ ]:
particle_torch_dynamic_axes = {
    "fs_data": {0: batch, 1: Dim.STATIC, 2: Dim.STATIC},
    "timestep": {0: batch},
    "mask": {0: batch, 1: Dim.STATIC, 2: Dim.STATIC},
    "ctxt_data": {0: batch, 1: Dim.STATIC, 2: Dim.STATIC},
    "ctxt_global_data": {0: batch, 1: Dim.STATIC},
}

In [ ]:
net_export = ONNXModel(
    model.net,
    dynamic_axes=particle_torch_dynamic_axes,
)

In [ ]:
import os

os.environ["TORCH_LOGS"] = "+dynamo"
os.environ["TORCHDYNAMO_VERBOSE"] = "1"

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net_export = net_export.to(device)
fs_data = fs_data.to(device)
timestep = timestep.to(device)
mask = mask.to(device)
ctxt_data = ctxt_data.to(device)
ctxt_global_data = ctxt_global_data.to(device)

In [ ]:
net_export.eval()
with torch.no_grad():
    export_program = torch.export.export(
        net_export,
        args=(fs_data, timestep, mask, ctxt_data, ctxt_global_data),
        dynamic_shapes=particle_torch_dynamic_axes,
    )

In [ ]:
print(export_program)

In [ ]:
torch.export.save(export_program, "exported_part_gpu.pt2")

In [ ]:
loaded_model = torch.export.load("exported_part_gpu.pt2")

In [ ]:
decomp_model = loaded_model.run_decompositions(torch.export.default_decompositions())

In [ ]:
from tqdm import tqdm

In [ ]:
for data in tqdm(dl):
    fs_data, ctxt_dict = data
    timestep = torch.rand(
        fs_data.shape[0], dtype=torch.float32
    )  # Random timestep for testing
    # mask = ctxt_dict["ctxt_mask"]
    mask = torch.stack([ctxt_dict["ctxt_mask"], ctxt_dict["fs_mask"]], dim=-1)
    ctxt_data = ctxt_dict["ctxt_data"]
    ctxt_global_data = ctxt_dict["ctxt_global_data"]
    fs_data = fs_data.to(device)
    timestep = timestep.to(device)
    mask = mask.to(device)
    ctxt_data = ctxt_data.to(device)
    ctxt_global_data = ctxt_global_data.to(device)

    outputs = loaded_model.module()(
        fs_data, timestep, mask, ctxt_data, ctxt_global_data
    )
    outputs_original = net_onnx(fs_data, timestep, mask, ctxt_data, ctxt_global_data)
    for o, o_orig in zip(outputs, outputs_original):
        abs_err = torch.abs(o - o_orig).max()
        if abs_err > 1e-6:
            print("Discrepancy found!")
            print(f"Max absolute error: {abs_err.item()}")
            print(f"Timestep: {timestep[0]}")
    break

In [ ]:
fs_data, ctxt_dict = next(iter(dl))
timestep = torch.rand(
    fs_data.shape[0], dtype=torch.float32
)  # Random timestep for testing
# mask = ctxt_dict["ctxt_mask"]
mask = torch.stack([ctxt_dict["ctxt_mask"], ctxt_dict["fs_mask"]], dim=-1)
ctxt_data = ctxt_dict["ctxt_data"]
ctxt_global_data = ctxt_dict["ctxt_global_data"]

In [ ]:
compiled_model = torch.compile(loaded_model.module())
compiled_original = torch.compile(net_onnx)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fs_data = fs_data.to(device)
timestep = timestep.to(device)
mask = mask.to(device)
ctxt_data = ctxt_data.to(device)
ctxt_global_data = ctxt_global_data.to(device)

In [ ]:
compiled_model(fs_data, timestep, mask, ctxt_data, ctxt_global_data)

In [ ]:
compiled_model = compiled_model.to(device)
compiled_original = compiled_original.to(device)
compiled_decomp = torch.compile(decomp_model.module()).to(device)

In [ ]:
print(decomp_model)

In [ ]:
# warm-up
compiled_model(fs_data, timestep, mask, ctxt_data, ctxt_global_data)
compiled_original(fs_data, timestep, mask, ctxt_data, ctxt_global_data)
compiled_decomp(fs_data, timestep, mask, ctxt_data, ctxt_global_data)

In [ ]:
%timeit compiled_model(fs_data, timestep, mask, ctxt_data, ctxt_global_data)

In [ ]:
%timeit compiled_original(fs_data, timestep, mask, ctxt_data, ctxt_global_data)

In [ ]:
%timeit compiled_decomp(fs_data, timestep, mask, ctxt_data, ctxt_global_data)

In [ ]:
%timeit loaded_model.module()(fs_data, timestep, mask, ctxt_data, ctxt_global_data)

In [ ]:
%timeit net_export(fs_data, timestep, mask, ctxt_data, ctxt_global_data)

In [ ]:
%timeit decomp_model(fs_data, timestep, mask, ctxt_data, ctxt_global_data)

In [ ]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 5)
        self.register_buffer("my_buffer", torch.ones(5), persistent=False)

    def forward(self, x):
        return self.linear(x) + self.my_buffer


model = MyModel()
example_inputs = (torch.randn(1, 10),)

# Export the model
exported_program = torch.export.export(model, example_inputs)

In [ ]:
exported_program.module().to(device)(example_inputs[0].to(device))